# Build district median household income file

This notebook creates a clean school-district income file from the NHGIS ACS extract.

**Source file**

`nhgis0001_ds272_20245_sd_uni.csv`

To download this exact file, navigate to https://data2.nhgis.org/.
- Select SD_UNI for geographic level
- Click "YEARS" then go to "ACS 5-Year Periods" and click "2020-2024
- Under data, click "B19013. Median Household Income in the Past 12 Months (in 2024 - Inflation-Adjusted Dollars)"
- To make things easier, restrict "Extent" to Texas

You will need an IPUMS account to download the dataset

**Source details from the NHGIS codebook**

- Dataset: 2024 American Community Survey 5-year estimates, covering 2020-2024
- Geography: School District (Unified)/Remainder by state
- Extent: Texas
- Table: Median Household Income in the Past 12 Months
- Universe: Households
- ACS table: `B19013`

**Columns used**

- `SDUNI`: school district name
- `SDUNIA`: school district code
- `AURUE001`: median household income in 2024 inflation-adjusted dollars
- `AURUM001`: margin of error for the income estimate

Output is `district_income.csv`, which keeps all Texas unified school districts.  

We will focus on San Antonio-area districts for the analysis portion

In [13]:
import pandas as pd
from pathlib import Path

## File paths

Repo is organized like this:

```text
staar-demographics/
├── data/
│   ├── raw/
│   │   └── nhgis0001_ds272_20245_sd_uni.csv
│   └── processed/
└── notebooks/
    └── 01_build_district_income.ipynb
```


In [14]:
RAW_DATA = Path("../data/raw/nhgis0001_ds272_20245_sd_uni.csv")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ALL_DISTRICTS_OUTPUT = PROCESSED_DIR / "district_income.csv"
SA_AREA_OUTPUT = PROCESSED_DIR / "sa_area_district_income.csv" 

## Load the NHGIS school district file



In [15]:
income_raw = pd.read_csv(
    RAW_DATA,
    dtype={
        "STATEA": "string",
        "COUNTYA": "string",
        "SDUNIA": "string",
    }
)

income_raw.head()

,GISJOIN,YEAR,STUSAB,REGIONA,DIVISIONA,STATE,STATEA,COUNTYA,COUSUBA,PLACEA,...,PCI,PUMAA,GEO_ID,BTTRA,BTBGA,TL_GEO_ID,NAME_E,AURUE001,NAME_M,AURUM001
0,G48000001,2020-2024,TX,NaN,NaN,Texas,48,<NA>,NaN,NaN,...,NaN,NaN,9700000US4800001,NaN,NaN,4800001.0,Crosbyton Consolidated Independent School Dist...,49135,Crosbyton Consolidated Independent School Dist...,15096
1,G48000002,2020-2024,TX,NaN,NaN,Texas,48,<NA>,NaN,NaN,...,NaN,NaN,9700000US4800002,NaN,NaN,4800002.0,"Spur Independent School District, Texas",48836,"Spur Independent School District, Texas",7517
2,G48000003,2020-2024,TX,NaN,NaN,Texas,48,<NA>,NaN,NaN,...,NaN,NaN,9700000US4800003,NaN,NaN,4800003.0,"Rocksprings Independent School District, Texas",38487,"Rocksprings Independent School District, Texas",24116
3,G48000005,2020-2024,TX,NaN,NaN,Texas,48,<NA>,NaN,NaN,...,NaN,NaN,9700000US4800005,NaN,NaN,4800005.0,Blackwell Consolidated Independent School Dist...,63942,Blackwell Consolidated Independent School Dist...,9982
4,G48000006,2020-2024,TX,NaN,NaN,Texas,48,<NA>,NaN,NaN,...,NaN,NaN,9700000US4800006,NaN,NaN,4800006.0,"Aspermont Independent School District, Texas",56779,"Aspermont Independent School District, Texas",22663


## Confirm the columns we need are present



In [16]:
required_columns = [
    "GISJOIN",
    "YEAR",
    "STATE",
    "SDUNI", #Distrct Name
    #For ambigious districts like Northside, the column has a parantheses with county name
    "SDUNIA",
    "AURUE001", #Median Household Income
    "AURUM001", #Margin of Error for Median Household Income
]

missing_columns = [column for column in required_columns if column not in income_raw.columns]

if missing_columns:
    raise ValueError(f"Missing expected NHGIS columns: {missing_columns}")

income_raw[required_columns].head()

,GISJOIN,YEAR,STATE,SDUNI,SDUNIA,AURUE001,AURUM001
0,G48000001,2020-2024,Texas,Crosbyton Consolidated Independent School Dist...,1,49135,15096
1,G48000002,2020-2024,Texas,Spur Independent School District,2,48836,7517
2,G48000003,2020-2024,Texas,Rocksprings Independent School District,3,38487,24116
3,G48000005,2020-2024,Texas,Blackwell Consolidated Independent School Dist...,5,63942,9982
4,G48000006,2020-2024,Texas,Aspermont Independent School District,6,56779,22663


## Build the clean district income file

A few choices here:

- Keep all Texas school districts, in case we want it later
- Keep `district_code`, even though it is different from TEA code
- Rename the NHGIS income fields 
- Keep the margin of error, so we avoid making conclusions about districts with large MOE


In [17]:
district_income = (
    income_raw[
        [
            "GISJOIN",
            "YEAR",
            "STATE",
            "SDUNI",
            "SDUNIA",
            "AURUE001",
            "AURUM001",
        ]
    ]
    .rename(
        columns={
            "GISJOIN": "gisjoin",
            "YEAR": "year",
            "STATE": "state",
            "SDUNI": "district",
            "SDUNIA": "district_code", #different than TEA code
            "AURUE001": "median_household_income",
            "AURUM001": "income_moe",
        }
    )
    .copy()
)

district_income["district"] = (
    district_income["district"]
    .str.strip()
    .str.replace(
        "Consolidated Independent School District",
        "CISD",
        regex=False,
    )
    .str.replace(
        "Municipal School District",
        "MSD",
        regex=False,
    )
    .str.replace(
        "Independent School District",
        "ISD",
        regex=False,
    )
    .str.replace(
        "Common School District",
        "CSD",
        regex=False,
    )
)

district_income["median_household_income"] = pd.to_numeric(
    district_income["median_household_income"],
    errors="raise",
)

district_income["income_moe"] = pd.to_numeric(
    district_income["income_moe"],
    errors="raise",
)

district_income.sample(n=5)

,gisjoin,year,state,district,district_code,median_household_income,income_moe
769,G48037080,2020-2024,Texas,Richland Springs ISD,37080,61976,21835
73,G48009810,2020-2024,Texas,Bellville ISD,9810,98899,7995
688,G48033780,2020-2024,Texas,Orangefield ISD,33780,84639,18127
315,G48019320,2020-2024,Texas,Florence ISD,19320,90000,13338
386,G48021960,2020-2024,Texas,Gunter ISD,21960,133000,28089


## Basic checks

Simple Checks
- Is this only Texas?
- Any district names missing?
- Any income estimates missing?
- Any district codes duplicated?


In [18]:
states = district_income["state"].drop_duplicates().sort_values().tolist()
states

['Texas']

In [19]:
if states != ["Texas"]:
    raise ValueError(f"Expected only Texas rows, but found: {states}")

if district_income["district"].isna().any():
    raise ValueError("Some rows are missing district names.")

if district_income["median_household_income"].isna().any():
    raise ValueError("Some rows are missing median household income estimates.")

duplicated_codes = district_income[district_income["district_code"].duplicated(keep=False)]

if not duplicated_codes.empty:
    raise ValueError("Some school district codes appear more than once. Check the NHGIS extract.")

district_income.shape

(1018, 7)

## Check for repeated district names




In [20]:
repeated_names = (
    district_income[district_income["district"].duplicated(keep=False)]
    .sort_values(["district"])
)

repeated_names[[
    "district",
    "district_code",
    "median_household_income",
]]

# No exact matches 
# This dataset will say Northside (Bexar) etc if there is any ambiguity

,district,district_code,median_household_income


## Export all Texas districts

This is the reusable file the STAAR notebook should use later.


In [21]:
district_income = district_income.sort_values("district").reset_index(drop=True)

district_income.to_csv(ALL_DISTRICTS_OUTPUT, index=False)

ALL_DISTRICTS_OUTPUT

PosixPath('../data/processed/district_income.csv')

## San Antonio-area quick file

File with just San Antonio-area districts

For districts with names that repeat statewide, we can use county names in parantheses to confirm

In [22]:
sa_area_income = district_income[
    district_income["district"].isin([
        "Alamo Heights ISD",
        "Boerne ISD",
        "Comal ISD",
        "East Central ISD",
        "Edgewood ISD (Bexar County)",
        "Fort Sam Houston ISD",
        "Harlandale ISD",
        "Judson ISD",
        "Lackland ISD",
        "Medina Valley ISD",
        "North East ISD",
        "Northside ISD (Bandera, Bexar, and Medina Counties)",
        "Randolph Field ISD",
        "San Antonio ISD",
        "Schertz-Cibolo-Universal City ISD",
        "Somerset ISD",
        "South San Antonio ISD",
        "Southside ISD",
        "Southwest ISD",
    ])
].sort_values("district").reset_index(drop=True)

assert len(sa_area_income) == 19

sa_area_income

#We can filter properly for Northside and Edgewood!

,gisjoin,year,state,district,district_code,median_household_income,income_moe
0,G48007590,2020-2024,Texas,Alamo Heights ISD,7590,100268,10037
1,G48010710,2020-2024,Texas,Boerne ISD,10710,136279,10659
2,G48014730,2020-2024,Texas,Comal ISD,14730,110926,3282
3,G48017850,2020-2024,Texas,East Central ISD,17850,80803,6282
4,G48018150,2020-2024,Texas,Edgewood ISD (Bexar County),18150,42055,1987
5,G48020160,2020-2024,Texas,Fort Sam Houston ISD,20160,100662,21360
6,G48022470,2020-2024,Texas,Harlandale ISD,22470,51927,3821
7,G48024990,2020-2024,Texas,Judson ISD,24990,76852,3281
8,G48026370,2020-2024,Texas,Lackland ISD,26370,64539,15330
9,G48030060,2020-2024,Texas,Medina Valley ISD,30060,99115,6604


## Export the San Antonio-area convenience file

Keeping districyt code and moe


In [23]:
sa_area_income[[
    "district",
    "district_code",
    "median_household_income",
    "income_moe",
]].to_csv(SA_AREA_OUTPUT, index=False)

SA_AREA_OUTPUT

PosixPath('../data/processed/sa_area_district_income.csv')

## Easy view of SA-area income

Note suburbs/ outer districts have six-figure incomes, while urban core reported sub-living wage (based on MIT calculator)

https://livingwage.mit.edu/metros/41700

Randolph, Fort Sam, and Lackland are all districts for military bases.
Any analysis should be sure to note this distinction.

Though there are low income families on bases, they get  
support and resources that civilian families do not.


In [24]:
sa_area_income[[
    "district",
    "median_household_income",
    "income_moe",
]].sort_values("median_household_income", ascending=False)

,district,median_household_income,income_moe
1,Boerne ISD,136279,10659
12,Randolph Field ISD,113250,22031
2,Comal ISD,110926,3282
14,Schertz-Cibolo-Universal City ISD,105726,3973
5,Fort Sam Houston ISD,100662,21360
0,Alamo Heights ISD,100268,10037
9,Medina Valley ISD,99115,6604
3,East Central ISD,80803,6282
11,"Northside ISD (Bandera, Bexar, and Medina Coun...",80406,1191
10,North East ISD,79074,1581
